<a href="https://colab.research.google.com/github/SheffieldML/GPc/blob/main/python/examples/basic_kernels_compound.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Basic kernels and compounds with `gpc`

GPy-tutorial style: build a compound covariance with `gpc.kern.*`, attach it to `GPRegression`, read/set parameters by path.

**CIP-0027.** Observation noise lives on `likelihood.variance` (alias `Gaussian_noise.variance`). `Bias` / `White` are opt-in kernel terms — do not confuse `White` with Gaussian observation noise.

GPy (conceptual)::

```python
k = GPy.kern.RBF(1) + GPy.kern.Bias(1) + GPy.kern.White(1)
m = GPy.models.GPRegression(X, Y, k)
```

`gpc` (signal + optional Bias; noise on the likelihood)::

```python
k = gpc.kern.RBF(1) + gpc.kern.Bias(1)
m = gpc.models.GPRegression(X, Y, kernel=k)
```

```bash
pip install matplotlib ipywidgets
```


In [ ]:
%pip install --no-cache-dir --upgrade --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "gpc==0.1.0a5"


In [ ]:
%pip install matplotlib ipywidgets


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import sys
from pathlib import Path

root = Path.cwd()
if (root / "gpc").is_dir():
    sys.path.insert(0, str(root))
elif (root.parent / "gpc").is_dir():
    sys.path.insert(0, str(root.parent))

import numpy as np
import gpc

rng = np.random.default_rng(1)
X = np.linspace(0.0, 1.0, 30)[:, None]
Y = 0.4 + np.sin(2 * np.pi * X) + 0.05 * rng.normal(size=X.shape)

k = gpc.kern.RBF(1) + gpc.kern.Bias(1)
m = gpc.models.GPRegression(X, Y, kernel=k)
m.set_by_path("rbf.lengthscale", 0.25)
m.set_by_path("rbf.variance", 1.0)
m.set_by_path("bias.variance", 0.25)
m.set_by_path("likelihood.variance", 0.05)
print("paths:", m.param_paths())
print(m)
m.optimize(iters=60, messages=True)
print("objective:", m.objective())
print(m)


In [ ]:
from gpc.plot import plot_1d
import matplotlib.pyplot as plt

# Compare compound vs signal-only string kernel on the same data.
m_sig = gpc.GPRegression(X, Y, kernel="rbf")
m_sig.set_by_path("rbf.lengthscale", 0.25)
m_sig.optimize(iters=60)

Xtest = np.linspace(-0.1, 1.1, 200)[:, None]
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
plot_1d(m, X, Y, Xtest=Xtest, ax=axes[0], label="RBF+Bias")
axes[0].set_title("kern.RBF + kern.Bias")
plot_1d(m_sig, X, Y, Xtest=Xtest, ax=axes[1], label="string rbf")
axes[1].set_title("kernel='rbf' (signal only)")
plt.tight_layout()
plt.show()
